# TV OFF Long-form — ACE-Step 1.5 + SDXL Turbo
Dry-run only. Generates 18 independent sections for a one-hour mix and one native 16:9 image. No YouTube upload.


In [ ]:
# TV OFF / ACE-Step 1.5 — production Original Music generator
# Colab's current base image is Python 3.13; upstream ACE-Step currently
# requires Python <3.13. We therefore create an isolated Python 3.12 venv.
import os, sys, json, subprocess, shutil, secrets
from pathlib import Path

WORK = Path("/content")
REPO = Path("/tmp/ACE-Step-1.5")
OUT = WORK / "ace_step_output"
OUT.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None):
    print("+", " ".join(map(str, cmd)), flush=True)
    child_env = os.environ.copy()
    # Colab's Jupyter process exports an inline Matplotlib backend that is
    # unavailable in the isolated Python 3.12 venv. Use a headless backend.
    child_env["MPLBACKEND"] = "Agg"
    result = subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, env=child_env, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n", flush=True)
    if result.stderr:
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n", file=sys.stderr, flush=True)
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, list(map(str, cmd)))

# Confirm that Colab has the expected CUDA hardware before installing packages.
import torch
print("Colab Python:", sys.version)
print("Colab PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; refusing CPU fallback.")
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
for i, name in enumerate(gpu_names):
    print("GPU", i, name, round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2), "GiB")
if not any("T4" in name for name in gpu_names):
    raise RuntimeError(f"Expected Tesla T4; detected {gpu_names}")

# Pin to the actual upstream latest release tag (v0.1.8), not a nonexistent v1.5.2 tag.
if not REPO.exists():
    run(["git", "clone", "--depth", "1", "--branch", "v0.1.8",
         "https://github.com/ace-step/ACE-Step-1.5.git", REPO])

# The upstream v0.1.8 release hardcodes float16 on pre-Ampere CUDA GPUs.
# Its NaN error hint mentions ACESTEP_DTYPE, but this release does not read
# that variable. Apply a guarded local patch to enable float32 for the T4.
orchestrator = REPO / "acestep/core/generation/handler/init_service_orchestrator.py"
orchestrator_text = orchestrator.read_text(encoding="utf-8")
old_dtype_block = '''            elif resolved_device == "cuda":
                if gpu_config.cuda_supports_bfloat16():
                    self.dtype = torch.bfloat16
                else:
                    self.dtype = torch.float16
                    logger.info(
                        "[initialize_service] Pre-Ampere CUDA detected: "
                        "using float16 instead of bfloat16."
                    )'''
new_dtype_block = '''            elif resolved_device == "cuda":
                forced_dtype = os.environ.get("ACESTEP_DTYPE", "").strip().lower()
                if forced_dtype == "float32":
                    self.dtype = torch.float32
                    logger.info("[initialize_service] ACESTEP_DTYPE=float32 override applied.")
                elif gpu_config.cuda_supports_bfloat16():
                    self.dtype = torch.bfloat16
                else:
                    self.dtype = torch.float16
                    logger.info(
                        "[initialize_service] Pre-Ampere CUDA detected: "
                        "using float16 instead of bfloat16."
                    )'''
if old_dtype_block not in orchestrator_text:
    raise RuntimeError("Expected v0.1.8 CUDA dtype block not found; refusing an unsafe source patch.")
orchestrator.write_text(orchestrator_text.replace(old_dtype_block, new_dtype_block, 1), encoding="utf-8")
os.environ["ACESTEP_DTYPE"] = "float32"
print("Applied guarded ACE-Step T4 float32 override:", orchestrator)

# ACE-Step declares Python >=3.11,<3.13. Colab's host is Python 3.13,
# so use an isolated 3.12 runtime instead of patching upstream metadata.
uv_path = shutil.which("uv")
if not uv_path:
    uv_install_dir = WORK / ".local" / "bin"
    uv_install_dir.mkdir(parents=True, exist_ok=True)
    run(["bash", "-lc", f"curl -LsSf https://astral.sh/uv/install.sh | env UV_INSTALL_DIR={uv_install_dir} sh"])
    uv_path = str(uv_install_dir / "uv")
uv = Path(uv_path)
if not uv.is_file():
    raise RuntimeError(f"uv executable was not installed; expected {uv}. PATH={os.environ.get('PATH')}")
run([uv, "python", "install", "3.12"])
venv = REPO / ".venv"
run([uv, "venv", "--python", "3.12", str(venv)], cwd=REPO)
python = venv / "bin" / "python"
run([uv, "pip", "install", "--python", str(python), "-e", "."], cwd=REPO)

# Generate using the isolated supported Python runtime.
# Fetch this PR branch's independent multi-section runner and execute it
# inside the already-installed Python 3.12 ACE-Step environment.
repo_checkout = WORK / "TV-OFF-AutoPublisher"
if repo_checkout.exists():
    shutil.rmtree(repo_checkout)
run(["git", "clone", "--depth", "1", "--branch", "feature/longform-production",
     "https://github.com/Rsychs/TV-OFF-AutoPublisher.git", repo_checkout])
run([python, repo_checkout / "longform" / "generate_sections_colab.py",
     "--hours", "1", "--output", OUT / "sections"])
print("LONGFORM ACE-STEP SECTIONS GENERATED.")


In [ ]:
# Generate one native 16:9 SDXL Turbo artwork for the assembled long-form mix.
import gc, hashlib, json, secrets, subprocess, sys
from pathlib import Path
import torch
from PIL import Image, ImageStat
OUT = Path("/content/ace_step_output")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA required for SDXL long-form artwork.")
if not any("T4" in torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())):
    raise RuntimeError("Expected Tesla T4 for SDXL artwork.")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "diffusers>=0.32.0", "transformers>=4.46.0", "accelerate>=1.1.0",
    "safetensors>=0.4.5", "Pillow>=10.0"], check=True)
from diffusers import AutoPipelineForText2Image
torch.cuda.empty_cache()
gc.collect()
seed = secrets.randbelow(2**31 - 1) + 1
prompt = ("Photorealistic cinematic panoramic 16:9 interior of a premium contemporary "
    "music listening lounge, sculptural walnut furniture, analog hi-fi equipment, "
    "subtle brass details, large architectural windows at blue hour, layered warm "
    "practical lighting, rich natural materials, atmospheric but realistic, "
    "high-end editorial interior photography, no people, no text, no logos, no watermark.")
pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo",
    torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
pipe.to("cuda")
try: pipe.enable_attention_slicing()
except Exception: pass
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0,
    width=1360, height=768,
    generator=torch.Generator(device="cuda").manual_seed(seed)).images[0]
image = image.convert("RGB").resize((1920,1080), Image.Resampling.LANCZOS)
path = OUT / "longform_visual_16x9.jpg"
image.save(path, "JPEG", quality=95, optimize=True)
if path.stat().st_size < 100000 or max(ImageStat.Stat(image).stddev) < 8:
    raise RuntimeError("Long-form SDXL visual failed image quality gate.")
manifest = {"status":"ok","model":"stabilityai/sdxl-turbo","seed":seed,
    "prompt":prompt,"visual_file":path.name,
    "visual_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),"width":1920,"height":1080}
(OUT / "longform_visual_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print("LONGFORM_SDXL_VISUAL_OK",json.dumps(manifest))


In [ ]:
# Package all unique source sections, manifests and native 16:9 artwork.
import hashlib, json, zipfile
from pathlib import Path
OUT=Path("/content/ace_step_output")
sm_path=OUT/"sections"/"longform_sections_manifest.json"
vm_path=OUT/"longform_visual_manifest.json"
if not sm_path.is_file() or not vm_path.is_file(): raise RuntimeError("Missing generation manifests.")
sm=json.loads(sm_path.read_text()); vm=json.loads(vm_path.read_text())
if sm.get("status")!="sections_generated" or sm.get("published") is not False: raise RuntimeError("Section generation manifest failed.")
if vm.get("status")!="ok" or (vm.get("width"),vm.get("height"))!=(1920,1080): raise RuntimeError("Native 16:9 visual manifest failed.")
for section in sm.get("sections",[]):
    folder=OUT/"sections"/section["directory"]; m=json.loads((folder/"ace_step_smoke_result.json").read_text())
    audio=folder/m["audio_file"]
    if not audio.is_file() or hashlib.sha256(audio.read_bytes()).hexdigest()!=m.get("sha256"): raise RuntimeError(f"Section hash validation failed: {folder.name}")
visual=OUT/vm["visual_file"]
if not visual.is_file() or hashlib.sha256(visual.read_bytes()).hexdigest()!=vm["visual_sha256"]: raise RuntimeError("Long-form visual hash mismatch.")
archive=Path("/content/tv_off_longform_dry_run.zip")
with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file() and p!=archive: z.write(p,arcname=p.relative_to(OUT))
print("LONGFORM_PACKAGE_OK",archive,"bytes=",archive.stat().st_size,"sections=",sm["section_count"])
